<a href="https://colab.research.google.com/github/johncase17/Data-Science-Project-1/blob/main/KPConv%2BUNetSeeded.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hybrid KPConv + BEV U-Net — Terrain Classification on SemanticKITTI


## 1. Setup

In [ ]:
import os, glob, random, time, warnings, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
from scipy.spatial import KDTree
from scipy.ndimage import uniform_filter
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import shutil

warnings.filterwarnings("ignore")
torch.manual_seed(42); random.seed(42); np.random.seed(42)

import torch.serialization
try:
    torch.serialization.add_safe_globals([np._core.multiarray.scalar])
except Exception:
    pass

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
WORK       = "/kaggle/working"
BASE       = "/kaggle/input/datasets/luischavarriazamora/semantic-kitti"
VEL_ROOT   = f"{BASE}/data_odometry_velodyne/dataset/sequences"
LBL_ROOT   = f"{BASE}/data_odometry_labels/dataset/sequences"
CACHE_DIR  = "/kaggle/temp/cache"
SCALES_DIR = "/kaggle/temp/scales"
os.makedirs(CACHE_DIR,  exist_ok=True)
os.makedirs(SCALES_DIR, exist_ok=True)

bins = sorted(glob.glob(f"{VEL_ROOT}/*/velodyne/*.bin"))
lbls = sorted(glob.glob(f"{LBL_ROOT}/*/labels/*.label"))
print(f"GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None - enable GPU'}")
print(f"Velodyne : {len(bins):,}   Labels: {len(lbls):,}")
assert bins and lbls, "Data not found"

CLASS_COLORS = {0:[.3,.3,.3], 1:[.2,.8,.2], 2:[1.,.8,.1], 3:[.9,.2,.1], 4:[.1,.3,.8]}
CLASS_NAMES  = {0:"Unknown", 1:"Flat/Traversable", 2:"Slope", 3:"Obstacle", 4:"Void"}
NUM_CLASSES  = 5

CFG = dict(
    img_size   = 256,
    num_classes= NUM_CLASSES,
    batch_size = 16,       # safe with 16GB VRAM
    epochs     = 20,
    lr         = 3e-4,
    device     = DEVICE,
    checkpoint = f"{WORK}/latest_checkpoint.pt",
    best_model = f"{WORK}/best_model.pt",
    history    = f"{WORK}/history.npy",
    patience   = 8,   # early stopping: stop if no improvement for 8 epochs
)

RAW_TO_TERRAIN = {
    40:1,44:1,48:1,49:1,60:1,72:2,
    10:3,11:3,13:3,15:3,16:3,18:3,20:3,
    30:3,31:3,32:3,50:3,51:3,52:3,70:3,71:3,80:3,81:3,99:4,
}

def remap_labels(raw):
    out = np.zeros_like(raw, dtype=np.int64)
    for rid, cls in RAW_TO_TERRAIN.items():
        out[raw==rid] = cls
    return out

def build_bev(pts, sz=256, res=0.2):
    if pts.shape[0]<10: return np.zeros((sz,sz,5),np.float32)
    x,y,z,intensity = pts[:,0],pts[:,1],pts[:,2],pts[:,3]
    x_min,y_min = x.min(),y.min()
    W = max(1,int(np.ceil((x.max()-x_min)/res))+1)
    H = max(1,int(np.ceil((y.max()-y_min)/res))+1)
    xi = np.clip(((x-x_min)/res).astype(np.int32),0,W-1)
    yi = np.clip(((y-y_min)/res).astype(np.int32),0,H-1)
    flat = yi*W+xi
    bev_max=np.full(H*W,-np.inf,np.float32); bev_min=np.full(H*W,np.inf,np.float32)
    bev_cnt=np.zeros(H*W,np.int32); bev_int=np.zeros(H*W,np.float32)
    np.maximum.at(bev_max,flat,z); np.minimum.at(bev_min,flat,z)
    np.add.at(bev_cnt,flat,1); np.add.at(bev_int,flat,intensity)
    bev_max=bev_max.reshape(H,W); bev_min=bev_min.reshape(H,W)
    bev_cnt=bev_cnt.reshape(H,W); bev_int=bev_int.reshape(H,W)
    mask=bev_cnt>0; bev_max[~mask]=0; bev_min[~mask]=0
    bev_rng=np.where(mask,bev_max-bev_min,0).astype(np.float32)
    z_vals=bev_max[mask]
    if len(z_vals)<2: return np.zeros((sz,sz,5),np.float32)
    z_lo=np.percentile(z_vals,2); z_hi=np.percentile(z_vals,98)
    def norm(a,lo,hi): return np.where(mask,np.clip((a-lo)/(hi-lo+1e-6),0,1),0).astype(np.float32)
    r_hi=bev_rng[mask].max() if mask.any() else 1.0
    ch_maxh=norm(bev_max,z_lo,z_hi); ch_minh=norm(bev_min,z_lo,z_hi)
    ch_rng=norm(bev_rng,0,r_hi)
    ch_dens=np.log1p(bev_cnt).astype(np.float32); ch_dens/=ch_dens.max()+1e-6
    ch_int=np.where(mask,bev_int/(bev_cnt+1e-6),0).astype(np.float32); ch_int/=ch_int.max()+1e-6
    bev=np.stack([ch_maxh,ch_minh,ch_rng,ch_dens,ch_int],axis=-1)
    h,w=bev.shape[:2]; ph,pw=max(0,sz-h),max(0,sz-w)
    bev=np.pad(bev,[(ph//2,ph-ph//2),(pw//2,pw-pw//2),(0,0)],mode="constant")
    h,w=bev.shape[:2]; sh,sw=(h-sz)//2,(w-sz)//2
    return bev[sh:sh+sz,sw:sw+sz].astype(np.float32)

def build_bev_labels(pts,labels,sz=256,res=0.2):
    x,y=pts[:,0],pts[:,1]; x_min,y_min=x.min(),y.min()
    W=max(1,int(np.ceil((x.max()-x_min)/res))+1)
    H=max(1,int(np.ceil((y.max()-y_min)/res))+1)
    xi=np.clip(((x-x_min)/res).astype(np.int32),0,W-1)
    yi=np.clip(((y-y_min)/res).astype(np.int32),0,H-1)
    flat=yi*W+xi; lbl_map=np.zeros(H*W,dtype=np.int64)
    for cls in [0,1,2,3,4]: lbl_map[flat[labels==cls]]=cls
    lbl_map=lbl_map.reshape(H,W); h,w=lbl_map.shape
    ph,pw=max(0,sz-h),max(0,sz-w)
    lbl_map=np.pad(lbl_map,[(ph//2,ph-ph//2),(pw//2,pw-pw//2)],mode="constant")
    h,w=lbl_map.shape; sh,sw=(h-sz)//2,(w-sz)//2
    return lbl_map[sh:sh+sz,sw:sw+sz]

VOXEL_SIZES=[0.04,0.08,0.16,0.32,0.64]

def py_grid_subsample(pts,voxel):
    vi=np.floor(pts[:,:3]/voxel).astype(np.int64)
    key=vi[:,0]*(2**40)+vi[:,1]*(2**20)+vi[:,2]
    _,inv,cnt=np.unique(key,return_inverse=True,return_counts=True)
    sub=np.zeros((cnt.shape[0],pts.shape[1]),np.float32)
    np.add.at(sub,inv,pts); return sub/cnt[:,None]

def py_radius_neighbors(q,s,radius,max_nn=16):
    tree=KDTree(s[:,:3]); idx_list=tree.query_ball_point(q[:,:3],radius)
    M=len(q); idx=np.zeros((M,max_nn),np.int64)
    for i,nbrs in enumerate(idx_list):
        if len(nbrs)==0: nbrs=[np.argmin(np.linalg.norm(s[:,:3]-q[i,:3],axis=1))]
        nbrs=np.array(nbrs[:max_nn]); idx[i,:len(nbrs)]=nbrs
        if len(nbrs)<max_nn: idx[i,len(nbrs):]=nbrs[-1]
    return idx

def precompute_scales(pts,max_pts=6000):
    if len(pts)>max_pts: pts=pts[np.random.choice(len(pts),max_pts,replace=False)]
    scales=[pts]
    for vs in VOXEL_SIZES[:-1]: scales.append(py_grid_subsample(scales[-1],vs))
    scales.append(py_grid_subsample(scales[-1],VOXEL_SIZES[-1]))
    nidx=[]
    for s in range(len(scales)-1):
        r=VOXEL_SIZES[s]*2.5
        nidx.append(torch.from_numpy(py_radius_neighbors(scales[s+1],scales[s],r,max_nn=16)).long())
    return [torch.from_numpy(sc).float() for sc in scales],nidx

_,_,free=shutil.disk_usage("/kaggle/temp")
cached =len([f for f in os.listdir(CACHE_DIR)  if f.endswith("_bev.npy")])
scaled =len([f for f in os.listdir(SCALES_DIR) if f.endswith("_scales.npz")])
print(f"/kaggle/temp free : {free/1e9:.1f} GB")
print(f"Cache  : {cached:,} scans")
print(f"Scales : {scaled:,} scans")
print(f"Setup complete | device={DEVICE} | batch_size={CFG['batch_size']} | epochs={CFG['epochs']}")


GPU      : Tesla T4
Velodyne : 43,552   Labels: 23,201
/kaggle/temp free : 1310.5 GB
Cache  : 0 scans
Scales : 0 scans
Setup complete | device=cuda | batch_size=16 | epochs=20


## 2. Cache — run once (~90 min), skips existing files

In [ ]:
sample_files=[f for f in os.listdir(CACHE_DIR) if f.endswith("_bev.npy")]
if sample_files:
    old_dtype=np.load(f"{CACHE_DIR}/{sample_files[0]}").dtype
    if old_dtype==np.float16:
        print("Old float16 cache — clearing to rebuild in uint8")
        shutil.rmtree(CACHE_DIR); os.makedirs(CACHE_DIR)

all_bins_c=[]
for seq in ["00","01","02","03","04","05","06","07","08","09","10"]:
    all_bins_c+=sorted(glob.glob(f"{VEL_ROOT}/{seq}/velodyne/*.bin"))

already_bev=len([f for f in os.listdir(CACHE_DIR)  if f.endswith("_bev.npy")])
already_sc =len([f for f in os.listdir(SCALES_DIR) if f.endswith("_scales.npz")])
print(f"Total: {len(all_bins_c):,}  BEV: {already_bev:,}  Scales: {already_sc:,}")

t0=time.time()
for bf in tqdm(all_bins_c):
    parts=bf.replace(VEL_ROOT,"").strip("/"); seq,_,fname=parts.split("/")
    key=f"{seq}_{fname.replace('.bin','')}"
    out_bev=f"{CACHE_DIR}/{key}_bev.npy"
    out_pts=f"{CACHE_DIR}/{key}_pts.npy"
    out_lbl=f"{CACHE_DIR}/{key}_lbl.npy"
    out_sc =f"{SCALES_DIR}/{key}_scales.npz"

    if os.path.exists(out_bev) and os.path.exists(out_sc):
        continue

    pts=np.fromfile(bf,dtype=np.float32).reshape(-1,4)
    lf=os.path.join(LBL_ROOT,seq,"labels",fname.replace(".bin",".label"))
    if os.path.exists(lf):
        raw=np.fromfile(lf,dtype=np.uint32)&0xFFFF
        lbl_arr=remap_labels(raw.astype(np.int64))
    else:
        lbl_arr=np.zeros(len(pts),dtype=np.int64)

    pts_s=pts[np.random.choice(len(pts),6000,replace=False)] if len(pts)>6000 else pts

    _,_,free=shutil.disk_usage("/kaggle/temp")
    if free<500*1e6: print(f"⚠️ Temp full at {key}"); break

    if not os.path.exists(out_bev):
        bev=build_bev(pts); lbl_map=np.clip(build_bev_labels(pts,lbl_arr),0,4)
        np.save(out_bev,(bev*255).clip(0,255).astype(np.uint8))
        np.save(out_pts,pts_s.astype(np.float16))
        np.save(out_lbl,lbl_map.astype(np.uint8))

    if not os.path.exists(out_sc):
        pts_list,nidx_list=precompute_scales(pts_s,max_pts=6000)
        np.savez_compressed(out_sc,
            **{f"p{i}":p.numpy().astype(np.float16) for i,p in enumerate(pts_list)},
            **{f"n{i}":n.numpy().astype(np.int32)   for i,n in enumerate(nidx_list)})

cached=len([f for f in os.listdir(CACHE_DIR)  if f.endswith("_bev.npy")])
scaled=len([f for f in os.listdir(SCALES_DIR) if f.endswith("_scales.npz")])
_,used,free=shutil.disk_usage("/kaggle/temp")
print(f"\nDone in {(time.time()-t0)/60:.1f} min")
print(f"BEV: {cached:,}  Scales: {scaled:,}")
print(f"/kaggle/temp: {used/1e9:.1f} GB used | {free/1e9:.1f} GB free")


Total: 23,201  BEV: 0  Scales: 0


  0%|          | 0/23201 [00:00<?, ?it/s]


Done in 100.7 min
BEV: 23,201  Scales: 23,201
/kaggle/temp: 7364.4 GB used | 1292.5 GB free


## 3. Class weights from actual data

In [ ]:
all_keys=[f.replace("_bev.npy","") for f in os.listdir(CACHE_DIR) if f.endswith("_bev.npy")]
print(f"Computing class frequencies from {min(500,len(all_keys))} scans...")

counts=np.zeros(NUM_CLASSES,np.float64)
for key in tqdm(random.sample(all_keys,min(500,len(all_keys))),desc="Counting"):
    lbl=np.load(f"{CACHE_DIR}/{key}_lbl.npy").astype(np.int64)
    for c in range(NUM_CLASSES): counts[c]+=(lbl==c).sum()

freq=counts/counts.sum()
median=np.median(freq[freq>0])
weights=np.where(freq>0,median/freq,0.0)
weights=np.clip(weights,0.05,20.0)
weights=weights/weights.mean()

CLASS_WEIGHTS=torch.tensor(weights,dtype=torch.float32,device=DEVICE)

print(f"\n{'Class':<22} {'Freq%':>8}  {'Weight':>8}")
print("-"*42)
for c in range(NUM_CLASSES):
    print(f"{CLASS_NAMES[c]:<22} {freq[c]*100:>7.2f}%  {weights[c]:>8.3f}")
print("\n Weights computed from real data")


Computing class frequencies from 500 scans...


Counting:   0%|          | 0/500 [00:00<?, ?it/s]


Class                     Freq%    Weight
------------------------------------------
Unknown                  83.35%     0.016
Flat/Traversable          7.63%     0.174
Slope                     2.34%     0.566
Obstacle                  6.56%     0.202
Void                      0.11%     4.042

 Weights computed from real data


## 4. Dataset & DataLoaders

In [ ]:
class CachedHybridDataset(Dataset):
    def __init__(self,keys,augment=False):
        self.keys=keys; self.augment=augment

    def __len__(self): return len(self.keys)

    def __getitem__(self,idx):
        key=self.keys[idx]
        bev    =np.load(f"{CACHE_DIR}/{key}_bev.npy").astype(np.float32)/255.0
        pts    =np.load(f"{CACHE_DIR}/{key}_pts.npy").astype(np.float32)
        lbl_map=np.clip(np.load(f"{CACHE_DIR}/{key}_lbl.npy").astype(np.int64),0,4)

        if self.augment:
            if random.random()>.5: bev,lbl_map=np.fliplr(bev).copy(),np.fliplr(lbl_map).copy()
            if random.random()>.5: bev,lbl_map=np.flipud(bev).copy(),np.flipud(lbl_map).copy()
            k=random.randint(0,3); bev,lbl_map=np.rot90(bev,k).copy(),np.rot90(lbl_map,k).copy()

        scales_path=f"{SCALES_DIR}/{key}_scales.npz"
        if os.path.exists(scales_path):
            sc=np.load(scales_path)
            n_sc=sum(1 for k in sc if k.startswith("p"))
            pts_list =[torch.from_numpy(sc[f"p{i}"].astype(np.float32)) for i in range(n_sc)]
            nidx_list=[torch.from_numpy(sc[f"n{i}"].astype(np.int64))   for i in range(n_sc-1)]
        else:
            pts_list,nidx_list=precompute_scales(pts,max_pts=3000)

        return {
            "bev"    :torch.from_numpy(bev.transpose(2,0,1)).float(),
            "labels" :torch.from_numpy(lbl_map).long(),
            "pts"    :pts_list,
            "nidx"   :nidx_list,
        }

def collate_fn(batch):
    return {
        "bev"    :torch.stack([b["bev"]    for b in batch]),
        "labels" :torch.stack([b["labels"] for b in batch]),
        "pts"    :[b["pts"]  for b in batch],
        "nidx"   :[b["nidx"] for b in batch],
    }

random.seed(42); random.shuffle(all_keys)
val_keys  =[k for k in all_keys if k.startswith("08_")]
train_keys=[k for k in all_keys if not k.startswith("08_")]
if not val_keys:
    split=int(len(all_keys)*0.8); train_keys=all_keys[:split]; val_keys=all_keys[split:]

print(f"Total: {len(all_keys):,}  Train: {len(train_keys):,}  Val: {len(val_keys):,}")

train_ds=CachedHybridDataset(train_keys,augment=True)
val_ds  =CachedHybridDataset(val_keys,  augment=False)

train_loader=DataLoader(train_ds,batch_size=CFG["batch_size"],shuffle=True,
                        num_workers=2,pin_memory=True,collate_fn=collate_fn)
val_loader  =DataLoader(val_ds,  batch_size=CFG["batch_size"],shuffle=False,
                        num_workers=2,pin_memory=True,collate_fn=collate_fn)

print(f"Train batches: {len(train_loader):,}  Val batches: {len(val_loader):,}")
t0=time.time(); _=train_ds[0]; st=time.time()-t0
est=st*len(train_keys)/2/60
print(f"Sample time: {st*1000:.0f}ms  Est epoch: ~{est:.0f} min  Est 30ep: ~{est*30/60:.1f}hrs")


Total: 23,201  Train: 19,130  Val: 4,071
Train batches: 1,196  Val batches: 255
Sample time: 26ms  Est epoch: ~4 min  Est 30ep: ~2.0hrs


## 5. Hybrid Model

**Key fix:** KPConv now processes every sample in the batch independently,
then uses global average pooling to get a per-sample 3D feature vector.
This is fused into the BEV bottleneck via a small MLP.


In [ ]:
# ── BEV U-Net — exact copy from notebookb73e2cf9d9.ipynb ─────────────────────
class DoubleConv(nn.Module):
    def __init__(self,ic,oc,drop=0.1):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(ic,oc,3,padding=1,bias=False),nn.BatchNorm2d(oc),nn.ReLU(True),
            nn.Dropout2d(drop),
            nn.Conv2d(oc,oc,3,padding=1,bias=False),nn.BatchNorm2d(oc),nn.ReLU(True))
    def forward(self,x): return self.net(x)

# ── KPConv blocks ──────────────────────────────────────────────────────────────
class KPConv(nn.Module):
    def __init__(self,in_ch,out_ch,n_kpts=15,radius=0.1):
        super().__init__()
        self.radius=radius
        self.W=nn.Parameter(torch.randn(n_kpts,in_ch,out_ch)*0.01)
        self.register_buffer("kpts",self._init(n_kpts))
    @staticmethod
    def _init(K):
        p=torch.randn(K,3); p=F.normalize(p,dim=1)*0.66; p[0]=0; return p
    def forward(self,q,s,feats,nidx):
        nidx=nidx.clamp(0,s.shape[0]-1)
        rel=(s[nidx,:3]-q[:,:3].unsqueeze(1))/(self.radius+1e-8)
        corr=(1-(rel.unsqueeze(2)-self.kpts).norm(dim=-1)).clamp(min=0)
        w=torch.einsum("mnk,mnc->mkc",corr,feats[nidx])
        return torch.einsum("mkc,kco->mo",w,self.W)

class KPConvBlock(nn.Module):
    def __init__(self,ic,oc,radius):
        super().__init__()
        mid=max(oc//2,8)
        self.bn0=nn.BatchNorm1d(ic); self.fc1=nn.Linear(ic,mid)
        self.bn1=nn.BatchNorm1d(mid); self.kpc=KPConv(mid,mid,radius=radius)
        self.bn2=nn.BatchNorm1d(mid); self.fc2=nn.Linear(mid,oc)
        self.bn3=nn.BatchNorm1d(oc)
        self.skip=nn.Linear(ic,oc) if ic!=oc else nn.Identity()
        self.act=nn.LeakyReLU(0.1,True)
    def forward(self,q,s,feats,nidx):
        x=self.act(self.bn1(self.fc1(self.act(self.bn0(feats)))))
        x=self.act(self.bn2(self.kpc(q,s,x,nidx)))
        x=self.bn3(self.fc2(x))
        sc=self.skip(feats)
        if sc.shape[0]!=x.shape[0]: sc=sc[nidx[:,0]]
        return self.act(x+sc)

# ── KPConv encoder — processes one point cloud, returns global feature ─────────
class KPConvEncoder(nn.Module):
    def __init__(self,out_dim=256):
        super().__init__()
        self.kp1=KPConvBlock(4,  32, 0.04)
        self.kp2=KPConvBlock(32, 64, 0.08)
        self.kp3=KPConvBlock(64, 128,0.16)
        self.kp4=KPConvBlock(128,256,0.32)
        self.kpb=KPConvBlock(256,out_dim,0.64)

    def forward(self,pts_list,nidx_list):
        # pts_list: list of 6 point tensors at different scales
        # nidx_list: list of 5 neighbor index tensors
        p0,p1,p2,p3,p4,p5 = [p.to(DEVICE) for p in pts_list[:6]]
        n0,n1,n2,n3,n4    = [n.to(DEVICE) for n in nidx_list[:5]]
        f1=self.kp1(p1,p0,p0,n0)
        f2=self.kp2(p2,p1,f1,n1)
        f3=self.kp3(p3,p2,f2,n2)
        f4=self.kp4(p4,p3,f3,n3)
        fb=self.kpb(p5,p4,f4,n4)
        return fb.mean(0)  # global average pool → (out_dim,)

# ── Hybrid model ───────────────────────────────────────────────────────────────
class HybridTerrainNet(nn.Module):
    FEATS=(32,64,128,256)

    def __init__(self,nc=5,kp_dim=256):
        super().__init__()
        # BEV encoder — exact match to notebookb73e2cf9d9.ipynb
        self.downs=nn.ModuleList(); self.pool=nn.MaxPool2d(2)
        ch=5
        for f in self.FEATS: self.downs.append(DoubleConv(ch,f)); ch=f
        self.bot=DoubleConv(ch,ch*2)   # 256 → 512

        # KPConv encoder
        self.kp_enc=KPConvEncoder(out_dim=kp_dim)

        # Fusion MLP: takes KPConv global features and conditions the bottleneck
        # FiLM-style: predict scale and shift for bottleneck feature map
        bot_ch=512
        self.film_fc=nn.Sequential(
            nn.Linear(kp_dim,bot_ch*2),
            nn.ReLU(True))

        # BEV decoder — exact match to notebookb73e2cf9d9.ipynb
        self.ups=nn.ModuleList(); ch=bot_ch
        for f in reversed(self.FEATS):
            self.ups.append(nn.ConvTranspose2d(ch,f,2,stride=2))
            self.ups.append(DoubleConv(f*2,f)); ch=f
        self.head=nn.Conv2d(ch,nc,1)

        for m in self.modules():
            if isinstance(m,(nn.Linear,nn.Conv2d)):
                nn.init.kaiming_normal_(m.weight,nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self,batch):
        bev=batch["bev"].to(DEVICE); B,_,H,W=bev.shape

        # BEV encoder
        skips=[]; x=bev
        for d in self.downs: x=d(x); skips.append(x); x=self.pool(x)
        bot=self.bot(x)   # (B, 512, H/16, W/16)
        skips=skips[::-1]

        # KPConv: process EACH sample in batch independently
        kp_feats=[]
        for b_idx in range(B):
            feat=self.kp_enc(batch["pts"][b_idx],batch["nidx"][b_idx])
            kp_feats.append(feat)
        kp_feats=torch.stack(kp_feats)   # (B, kp_dim)

        # FiLM conditioning: scale and shift the bottleneck
        film=self.film_fc(kp_feats)            # (B, 512*2)
        gamma,beta=film.chunk(2,dim=1)         # each (B, 512)
        gamma=gamma.view(B,512,1,1)
        beta =beta.view(B,512,1,1)
        bot=bot*(1+gamma)+beta                 # FiLM: y = gamma*x + beta

        # BEV decoder
        x=bot
        for i in range(0,len(self.ups),2):
            x=self.ups[i](x); s=skips[i//2]
            if x.shape!=s.shape:
                x=F.interpolate(x,size=s.shape[2:],mode="bilinear",align_corners=False)
            x=self.ups[i+1](torch.cat([s,x],1))
        return self.head(x)

model=HybridTerrainNet(nc=NUM_CLASSES).to(DEVICE)
n_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"HybridTerrainNet : {n_params/1e6:.2f}M parameters")
print(f"Architecture: BEV U-Net + KPConv encoder with FiLM fusion")
print(f"KPConv processes EVERY sample in batch independently")


HybridTerrainNet : 8.78M parameters
Architecture: BEV U-Net + KPConv encoder with FiLM fusion
KPConv processes EVERY sample in batch independently


## 6. Loss, Optimiser & Metrics

In [ ]:
criterion=nn.CrossEntropyLoss(weight=CLASS_WEIGHTS,ignore_index=-1)
optimizer=torch.optim.AdamW(model.parameters(),lr=CFG["lr"],weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=CFG["epochs"],eta_min=1e-6)
scaler=torch.amp.GradScaler("cuda")   # AMP for faster training

def print_miou_report(iou,miou,accuracy=None,title="HYBRID MODEL"):
    print(f"\n{'='*50}")
    print(f"  MODEL ACCURACY REPORT — {title}")
    print(f"{'='*50}")
    if accuracy is not None: print(f"  Overall Pixel Accuracy : {accuracy:.2f}%")
    print(f"  Mean IoU (mIoU)        : {miou:.3f} ({miou*100:.1f}%)")
    print(f"{'─'*50}")
    for c in range(NUM_CLASSES):
        bar="█"*int(iou[c]*20)
        print(f"  {CLASS_NAMES[c]:<22} {iou[c]:.3f}  {bar}")
    print(f"{'─'*50}")
    print(f"  {'mIoU':<22} {miou:.3f}  ({miou*100:.1f}%)")
    print(f"{'='*50}")
    if accuracy: print(f"  Pixel Accuracy: {accuracy:.1f}%")

print(f"CrossEntropyLoss + AMP | AdamW lr={CFG['lr']} | CosineAnnealingLR")
print(f"Class weights: {[round(float(w),3) for w in CLASS_WEIGHTS.cpu()]}")


CrossEntropyLoss + AMP | AdamW lr=0.0003 | CosineAnnealingLR
Class weights: [0.016, 0.174, 0.566, 0.202, 4.042]


## 7. Training Loop

In [ ]:
# scaler defined in Cell 6 (Loss cell) — must run that first
# Redefine scaler here in case Cell 6 was not run
if "scaler" not in dir():
    scaler=torch.amp.GradScaler("cuda")

start_epoch=1
history={"train_loss":[],"val_loss":[],"train_miou":[],"val_miou":[]}
best_miou=0.0; best_epoch=0

if os.path.exists(CFG["checkpoint"]):
    ckpt=torch.load(CFG["checkpoint"],map_location=DEVICE,weights_only=False)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_epoch=ckpt["epoch"]+1
    best_miou=ckpt["val_miou"]; best_epoch=ckpt["epoch"]
    if "history" in ckpt: history=ckpt["history"]
    print(f"Resuming from epoch {start_epoch}  (best mIoU: {best_miou:.4f} @ ep {best_epoch})")
else:
    print("Starting fresh training...")

print(f"Training epochs {start_epoch}–{CFG['epochs']}")
print(f"Train: {len(train_loader):,} batches   Val: {len(val_loader):,} batches")
print("-"*76)
print(f"{'Ep':>3}  {'T-Loss':>8}  {'T-mIoU':>8}  {'V-Loss':>8}  {'V-mIoU':>8}  {'LR':>10}  {'Min':>4}")
print("-"*76)

for epoch in range(start_epoch,CFG["epochs"]+1):
    t_ep=time.time()

    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    tl=[]; itr=np.zeros(NUM_CLASSES,np.float64); utr=np.zeros(NUM_CLASSES,np.float64)
    for batch in tqdm(train_loader,desc=f"Ep{epoch:02d} train",leave=False):
        labels=batch["labels"].to(DEVICE)
        with torch.amp.autocast("cuda"):
            logits=model(batch)
            loss=criterion(logits,labels)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optimizer); scaler.update()
        tl.append(loss.item())
        p=logits.argmax(1).detach().cpu().numpy().flatten()
        t=labels.cpu().numpy().flatten()
        for c in range(NUM_CLASSES):
            itr[c]+=((p==c)&(t==c)).sum(); utr[c]+=((p==c)|(t==c)).sum()
        del logits,loss,p,t,labels; torch.cuda.empty_cache()
    tr_miou=float((itr/(utr+1e-8)).mean())

    # ── Val ───────────────────────────────────────────────────────────────────
    model.eval()
    vl=[]; iva=np.zeros(NUM_CLASSES,np.float64); uva=np.zeros(NUM_CLASSES,np.float64)
    with torch.no_grad():
        for batch in tqdm(val_loader,desc=f"Ep{epoch:02d} val  ",leave=False):
            labels=batch["labels"].to(DEVICE)
            with torch.amp.autocast("cuda"):
                logits=model(batch)
                loss=criterion(logits,labels)
            vl.append(loss.item())
            p=logits.argmax(1).cpu().numpy().flatten()
            t=batch["labels"].numpy().flatten()
            for c in range(NUM_CLASSES):
                iva[c]+=((p==c)&(t==c)).sum(); uva[c]+=((p==c)|(t==c)).sum()
            del logits,loss,p,t,labels; torch.cuda.empty_cache()

    va_iou=iva/(uva+1e-8); va_miou=float(va_iou.mean())
    scheduler.step()

    history["train_loss"].append(np.mean(tl)); history["val_loss"].append(np.mean(vl))
    history["train_miou"].append(tr_miou);     history["val_miou"].append(va_miou)

    torch.save({"epoch":epoch,"model":model.state_dict(),"optimizer":optimizer.state_dict(),
                "val_miou":va_miou,"class_iou":va_iou.tolist(),"history":history},
               CFG["checkpoint"])

    marker=""
    if va_miou>best_miou:
        best_miou=va_miou; best_epoch=epoch; marker=" ★"
        torch.save({"epoch":epoch,"model":model.state_dict(),
                    "val_miou":va_miou,"class_iou":va_iou.tolist()},CFG["best_model"])

    ep_min=(time.time()-t_ep)/60
    print(f"{epoch:>3}  {np.mean(tl):>8.4f}  {tr_miou:>8.4f}  "
          f"{np.mean(vl):>8.4f}  {va_miou:>8.4f}  "
          f"{optimizer.param_groups[0]['lr']:>10.2e}  {ep_min:>3.0f}m{marker}")
    torch.cuda.empty_cache(); gc.collect()

    # Early stopping
    if epoch - best_epoch >= CFG["patience"]:
        print(f"Early stopping at epoch {epoch} — no improvement for {CFG['patience']} epochs")
        break

print("-"*76)
print(f"Best val mIoU: {best_miou:.4f} @ epoch {best_epoch}")
np.save(CFG["history"],history)


Starting fresh training...
Training epochs 1–20
Train: 1,196 batches   Val: 255 batches
----------------------------------------------------------------------------
 Ep    T-Loss    T-mIoU    V-Loss    V-mIoU          LR   Min
----------------------------------------------------------------------------


Ep01 train:   0%|          | 0/1196 [00:00<?, ?it/s]

Ep01 val  :   0%|          | 0/255 [00:00<?, ?it/s]

  1    0.5371    0.5416    0.3692    0.7014    2.98e-04   14m ★


Ep02 train:   0%|          | 0/1196 [00:00<?, ?it/s]

...

Ep19 val  :   0%|          | 0/255 [00:00<?, ?it/s]

 19    0.1487    0.7657    0.4177    0.7326    2.84e-06   14m


Ep20 train:   0%|          | 0/1196 [00:00<?, ?it/s]

Ep20 val  :   0%|          | 0/255 [00:00<?, ?it/s]

 20    0.1481    0.7661    0.3936    0.7392    1.00e-06   14m
----------------------------------------------------------------------------
Best val mIoU: 0.7427 @ epoch 15


## 8. Final mIoU Report

In [ ]:
ckpt=torch.load(CFG["best_model"],map_location=DEVICE,weights_only=False)
best_epoch=ckpt["epoch"]
model.load_state_dict(ckpt["model"])
print(f"Best model: epoch {ckpt['epoch']}  val mIoU={ckpt['val_miou']:.4f}")

model.eval()
inter=np.zeros(NUM_CLASSES,np.float64); union=np.zeros(NUM_CLASSES,np.float64)
tp_all=0; total=0

with torch.no_grad():
    for batch in tqdm(val_loader,desc="Final eval"):
        labels=batch["labels"].to(DEVICE)
        with torch.amp.autocast("cuda"):
            logits=model(batch)
        pred=logits.argmax(1).cpu().numpy().flatten()
        true=batch["labels"].numpy().flatten()
        tp_all+=(pred==true).sum(); total+=len(true)
        for c in range(NUM_CLASSES):
            inter[c]+=((pred==c)&(true==c)).sum()
            union[c]+=((pred==c)|(true==c)).sum()
        del logits,labels; torch.cuda.empty_cache()

iou=inter/(union+1e-8); miou=float(iou.mean()); accuracy=tp_all/total*100
print_miou_report(iou,miou,accuracy)


print(f"  Hybrid     mIoU={miou:.3f}  Acc={accuracy:.2f}%")



Best model: epoch 15  val mIoU=0.7427


Final eval:   0%|          | 0/255 [00:00<?, ?it/s]


  MODEL ACCURACY REPORT — HYBRID MODEL
  Overall Pixel Accuracy : 98.43%
  Mean IoU (mIoU)        : 0.743 (74.3%)
──────────────────────────────────────────────────
  Unknown                0.999  ███████████████████
  Flat/Traversable       0.912  ██████████████████
  Slope                  0.774  ███████████████
  Obstacle               0.856  █████████████████
  Void                   0.172  ███
──────────────────────────────────────────────────
  mIoU                   0.743  (74.3%)
  Pixel Accuracy: 98.4%
  Hybrid     mIoU=0.743  Acc=98.43%


## 9. Training Curves & Per-class IoU

In [ ]:
# Load best checkpoint to get best_epoch and iou if not already defined
if "best_epoch" not in dir() or "iou" not in dir():
    _ckpt = torch.load(CFG["best_model"], map_location=DEVICE, weights_only=False)
    best_epoch = _ckpt["epoch"]
    iou = np.array(_ckpt["class_iou"])
    miou = float(iou.mean())
    print(f"Loaded from checkpoint: epoch {best_epoch}  mIoU={miou:.4f}")

if os.path.exists(CFG["history"]):
    history=np.load(CFG["history"],allow_pickle=True).item()

epochs_x=list(range(1,len(history["train_loss"])+1))
fig,axes=plt.subplots(1,2,figsize=(14,5))
for ax,k1,k2,title in zip(axes,
        ["train_loss","train_miou"],["val_loss","val_miou"],["Loss","mIoU"]):
    ax.plot(epochs_x,history[k1],label="Train",lw=2)
    ax.plot(epochs_x,history[k2],label="Val",  lw=2,linestyle="--")
    if best_epoch>0 and best_epoch<=len(epochs_x):
        ax.axvline(best_epoch,color="gold",lw=1.5,linestyle=":",label=f"Best ep {best_epoch}")
    ax.set_xlabel("Epoch"); ax.set_title(title); ax.legend(); ax.grid(alpha=.3)
plt.suptitle("Hybrid KPConv + BEV U-Net — Training History",fontsize=13)
plt.tight_layout()
plt.savefig(f"{WORK}/training_curves.png",dpi=130,bbox_inches="tight")
plt.show(); plt.close()

fig,ax=plt.subplots(figsize=(10,4))
colors=[CLASS_COLORS[i] for i in range(NUM_CLASSES)]
bars=ax.bar([CLASS_NAMES[i] for i in range(NUM_CLASSES)],iou,
            color=colors,edgecolor="white",linewidth=0.8)
for bar,v in zip(bars,iou):
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.01,
            f"{v:.3f}",ha="center",fontsize=10,fontweight="bold")
ax.axhline(miou,  color="black", linestyle="--",lw=1.5,label=f"Hybrid mIoU={miou:.3f}")
ax.axhline(0.749, color="blue",  linestyle=":", lw=1.5,label="BEV U-Net (0.749)")
ax.axhline(0.561, color="orange",linestyle=":", lw=1.5,label="KPConv (0.561)")
ax.set_ylim(0,1.1); ax.legend(); ax.grid(axis="y",alpha=.3)
ax.set_title(f"Per-class IoU — best epoch {best_epoch}")
plt.tight_layout()
plt.savefig(f"{WORK}/per_class_iou.png",dpi=130,bbox_inches="tight")
plt.show(); plt.close()
print("Plots saved")


Plots saved


## 10. Qualitative Results + Navigation

In [ ]:
def traversability(seg):
    t=np.zeros_like(seg,dtype=np.float32)
    t[seg==1]=1.0; t[seg==2]=0.5; return t

def visualise(bev_np,seg,title=""):
    trav=traversability(seg); H,W=trav.shape
    colour=np.zeros((H,W,3),np.float32)
    for c,col in CLASS_COLORS.items(): colour[seg==c]=col
    oy=int(H*.92); lah=int(H*.45)
    cs=uniform_filter(trav[max(0,oy-lah):oy,:].mean(0),size=max(1,W//8))
    wpx=int(cs.argmax()); sz2=W//2
    direc=("LEFT" if wpx<sz2-sz2//4 else "RIGHT" if wpx>sz2+sz2//4 else "STRAIGHT")
    ts=float(trav[oy-lah//2,wpx])
    status=("PATH CLEAR" if ts>=.8 else "CAUTION" if ts>=.4 else "BLOCKED")

    fig,axes=plt.subplots(1,3,figsize=(18,6))
    fig.patch.set_facecolor("#0d1117")
    for ax in axes: ax.set_facecolor("#0d1117")
    axes[0].imshow(bev_np[:,:,0],origin="lower",cmap="viridis")
    axes[0].set_title("BEV max-height",color="white",fontsize=11); axes[0].axis("off")
    axes[1].imshow(colour,origin="lower")
    axes[1].legend(handles=[mpatches.Patch(color=col,label=CLASS_NAMES[i])
                             for i,col in CLASS_COLORS.items()],
                   loc="upper right",fontsize=8,facecolor="#1a1a2e",labelcolor="white")
    axes[1].set_title("Segmentation",color="white",fontsize=11); axes[1].axis("off")
    axes[2].imshow(colour,origin="lower")
    ox=W//2; oyp=int(H*.08)
    axes[2].annotate("",xy=(wpx,int(H*.55)),xytext=(ox,oyp),
        arrowprops=dict(arrowstyle="->,head_width=0.8,head_length=0.5",color="lime",lw=4))
    axes[2].plot(ox,oyp,"w^",ms=14,markeredgecolor="black",markeredgewidth=1.5,zorder=5)
    axes[2].text(W//2,H*.95,f"➡ {direc}",color="white",fontsize=16,fontweight="bold",
                 ha="center",va="top",bbox=dict(boxstyle="round,pad=0.4",facecolor="#1a1a2e",alpha=.8))
    axes[2].text(W//2,H*.88,status,fontsize=13,fontweight="bold",ha="center",va="top",
                 color=("lime" if "CLEAR" in status else "yellow" if "CAUTION" in status else "red"),
                 bbox=dict(boxstyle="round,pad=0.3",facecolor="#1a1a2e",alpha=.8))
    axes[2].set_title("Navigation",color="white",fontsize=11); axes[2].axis("off")
    plt.suptitle(title,color="white",fontsize=11,y=1.01); plt.tight_layout()
    plt.savefig(f"{WORK}/qual_{title.replace('/','_')}.png",dpi=120,
                bbox_inches="tight",facecolor=fig.get_facecolor())
    plt.show(); plt.close()
    print(f"  {direc}  |  {status}  |  score={ts:.2f}")

model.eval()
for key in random.sample(val_keys,min(5,len(val_keys))):
    bev=np.load(f"{CACHE_DIR}/{key}_bev.npy").astype(np.float32)/255.0
    scales_path=f"{SCALES_DIR}/{key}_scales.npz"
    if os.path.exists(scales_path):
        sc=np.load(scales_path); n_sc=sum(1 for k in sc if k.startswith("p"))
        pl=[torch.from_numpy(sc[f"p{i}"].astype(np.float32)) for i in range(n_sc)]
        ni=[torch.from_numpy(sc[f"n{i}"].astype(np.int64))   for i in range(n_sc-1)]
    else:
        pts=np.load(f"{CACHE_DIR}/{key}_pts.npy").astype(np.float32)
        pl,ni=precompute_scales(pts,max_pts=3000)
    mb={"bev":torch.from_numpy(bev.transpose(2,0,1)).float().unsqueeze(0).to(DEVICE),
        "labels":torch.zeros(1,256,256,dtype=torch.long),
        "pts":[pl],"nidx":[ni]}
    with torch.no_grad():
        with torch.amp.autocast("cuda"):
            seg=model(mb).argmax(1).squeeze().cpu().numpy()
    del mb; torch.cuda.empty_cache()
    visualise(bev,seg,title=key)


  STRAIGHT  |  BLOCKED  |  score=0.00
  STRAIGHT  |  PATH CLEAR  |  score=1.00
  STRAIGHT  |  BLOCKED  |  score=0.00
  STRAIGHT  |  BLOCKED  |  score=0.00
  LEFT  |  BLOCKED  |  score=0.00
